In [44]:
%pip install datasets evaluate torch transformers huggingface_hub "evaluate[code_eval]"

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Looking in indexes: https://artifactory.tcsbank.ru/artifactory/api/pypi/python-all/simple

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [45]:
from datasets import load_dataset
from evaluate import load as load_metric

ds = load_dataset("openai_humaneval")

In [46]:
print(ds)

DatasetDict({
    test: Dataset({
        features: ['task_id', 'prompt', 'canonical_solution', 'test', 'entry_point'],
        num_rows: 164
    })
})


In [47]:
ds['test'].select(range(2))

Dataset({
    features: ['task_id', 'prompt', 'canonical_solution', 'test', 'entry_point'],
    num_rows: 2
})

In [48]:
import os
os.environ["HF_ALLOW_CODE_EVAL"] = "1"

In [49]:
from datasets import load_dataset
import evaluate

# 1) Load dataset
ds = load_dataset("openai_humaneval")  # usually only a "test" split

# 2) Load the correct measurement
code_eval = evaluate.load("code_eval", module_type="metric")

# 3) Build references
# references = [
#     {"task_id": ex["task_id"], "test": ex["test"], "entry_point": ex["entry_point"]}
#     for ex in ds["test"]
# ]

# references = [
#     ex["test"]
#     for ex in ds["test"]
# ]

# # 4) Example predictions: use canonical solutions (sanity check)
# #    Each element is a list of candidate programs for that task.
# predictions = [[ex["prompt"] + ex["canonical_solution"]] for ex in ds["test"]]

# # 5) Compute pass@1
# results = code_eval.compute(
#     references=references,
#     predictions=predictions,
#     k=[1],
#     timeout=3.0,
#     num_workers=4,
# )
# print(results)

In [50]:
import getpass
from huggingface_hub import login

token = getpass.getpass("Enter your Hugging Face token: ")

login(token=token)

Enter your Hugging Face token:  ········


In [56]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

device = 'cuda'
model_name = "google/gemma-7b"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, use_auth_token=True)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16 if torch.cuda.is_available() else None, use_auth_token=True).to(device)

# gen = pipeline(
#     "text-generation",
#     model=model_name,
#     # use_auth_token=True
#     # device_map="auto",
# )

def generate_candidates(prompt, n=5, max_new_tokens=512, temperature=0.2, top_p=0.95):
    
    # outputs = gen(
    #     prompt,
    #     do_sample=True,
    #     temperature=temperature,
    #     top_p=top_p,
    #     num_return_sequences=n,
    #     max_new_tokens=max_new_tokens,
    #     eos_token_id=tokenizer.eos_token_id,
    #     pad_token_id=tokenizer.pad_token_id,
    # )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, num_return_sequences=n, do_sample=True, temperature=temperature, eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id).to(device)
    outputs = tokenizer.batch_decode(outputs_ids, skip_special_tokens=True)
    
    completions = []
    for o in outputs:
        text = o
        
        if text.startswith(prompt):
            completion = text[len(prompt):]
        else:
            completion = text
        program = prompt + completion
        completions.append(program)
    return completions


subset = ds["test"].shuffle().select(range(33))

references = [
    ex["test"]
    for ex in subset
]

# Generate multiple candidates per task for pass@k
n_samples_per_task = 1
predictions = []
for ex in subset:
    candidates = generate_candidates(ex["prompt"], n=n_samples_per_task)
    predictions.append(candidates)


results = code_eval.compute(
    references=references,
    predictions=predictions,
    k=[n_samples_per_task],
    timeout=5.0,
    num_workers=16,
    # details=False,
)
print(results)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

({'pass@5': 0.7575757575757576}, defaultdict(<class 'list'>, {0: [(0, {'task_id': 0, 'passed': False, 'result': "failed: '[' was never closed (<string>, line 63)", 'completion_id': 0}), (1, {'task_id': 0, 'passed': False, 'result': "failed: '(' was never closed (<string>, line 62)", 'completion_id': 1}), (2, {'task_id': 0, 'passed': False, 'result': 'failed: unterminated triple-quoted string literal (detected at line 72) (<string>, line 55)', 'completion_id': 2}), (3, {'task_id': 0, 'passed': False, 'result': 'failed: expected an indented block after function definition on line 62 (<string>, line 63)', 'completion_id': 3}), (4, {'task_id': 0, 'passed': False, 'result': 'failed: expected an indented block after function definition on line 62 (<string>, line 63)', 'completion_id': 4})], 2: [(0, {'task_id': 2, 'passed': True, 'result': 'passed', 'completion_id': 0}), (1, {'task_id': 2, 'passed': True, 'result': 'passed', 'completion_id': 1}), (2, {'task_id': 2, 'passed': True, 'result': '

In [88]:
# import gc
# del model
# del tokenizer
# del outputs_ids
# del results
torch.cuda.empty_cache()
gc.collect()

0

In [57]:
results[0]['pass@5']

0.7575757575757576

In [43]:
print('def get_positive(l: list):\n    """Return only positive numbers in the list.\n    >>> get_positive([-1, 2, -4, 5, 6])\n    [2, 5, 6]\n    >>> get_positive([5, 3, -5, 2, -3, 3, 9, 0, 123, 1, -10])\n    [5, 3, 2, 3, 9, 123, 1]\n    """\n    return [x for x in l if x > 0]')

def get_positive(l: list):
    """Return only positive numbers in the list.
    >>> get_positive([-1, 2, -4, 5, 6])
    [2, 5, 6]
    >>> get_positive([5, 3, -5, 2, -3, 3, 9, 0, 123, 1, -10])
    [5, 3, 2, 3, 9, 123, 1]
    """
    return [x for x in l if x > 0]
